# Chapter 7: Search In-Depth

## Topic 2: Dense Retrieval — Recap and Its Specific Weaknesses

---

### 1. Concept and Intuition

**What dense retrieval is:**

- Every piece of text — a query, a document chunk, a policy clause — gets mapped to a single fixed-size vector of numbers
- The mapping is learned during the embedding model's training on millions of text pairs
- Texts that mean the same thing in different words → vectors pointing in nearly the same direction
- Texts that mean different things → vectors pointing in very different directions
- "Similar meaning" becomes "small angle between vectors" — something geometry can operate on
- This is the fundamental insight that BM25 completely lacks: BM25 cannot know that "exit FD early" and "premature withdrawal" mean the same thing; a dense embedding model can

**Why it exists — the one thing BM25 cannot do:**

- BM25 score = 0 when zero vocabulary overlap between query and document
- A customer writes "mera paisa fasa hua hai" — BM25 returns score = 0 for a policy chunk saying "premature closure penalty", because not a single word overlaps
- Dense retrieval returns a meaningful similarity score for this pair, because both texts encode the same semantic situation — money being held, restricted from access — regardless of which language or phrasing expresses it
- This is not a hypothetical: 64.4% of your 2,500 emails are Hinglish, and your policy documents are in English → vocabulary mismatch is the dominant failure mode in your corpus, not a rare edge case

**This is not new to your project:**

- Chapter 3's `2_Embedding_Semantic_Search.ipynb` already built this in full
- `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensions) running locally on CPU
- `model.encode(texts, normalize_embeddings=True)` → batch embedding
- `np.dot(a, b)` → cosine similarity (equal to dot product for normalised vectors)
- The in-memory store from Chapter 3 is brute-force dense retrieval: embed all chunks, embed the query, compute similarity against every chunk, return top-k

---

### 2. Internal Working — Step by Step

**Step 1: Model architecture (what happens inside the embedding model):**

- `paraphrase-multilingual-MiniLM-L12-v2` is a BERT-style encoder with 12 transformer layers
- Input: tokenized text → WordPiece tokens with position embeddings
- Processing: 12 layers of bidirectional self-attention — each token attends to every other token in both directions simultaneously (this is the key difference from GPT-style decoders)
- Output: one 384-dimensional vector per token in the sequence
- Pooling: mean pooling across all token vectors → one 384-dimensional vector per input text
- This single vector is the embedding — it encodes the overall meaning of the entire input text

**Step 2: Normalisation and why it matters:**

```python
model.encode(texts, normalize_embeddings=True)
```

- Scales every output vector to unit length (L2 norm = 1)
- Makes cosine similarity equal to dot product: `cos(a,b) = dot(a,b) / (||a|| × ||b||)` → when `||a|| = ||b|| = 1` → `cos(a,b) = dot(a,b)`
- Your notebook confirmed this: `np.isclose(np.dot(vec_fd_a, vec_fd_b), cosine_similarity(vec_fd_a, vec_fd_b)) == True`
- Qdrant (Chapter 6) exploits this same shortcut internally when `distance=Distance.COSINE` is set on normalised vectors

**Step 3: Offline index building (at ingest time):**

- Every chunk from the 17 knowledge base pages goes through `model.encode()`
- The resulting 384-dimensional normalised vector, along with `page_content` and `metadata`, is upserted into Qdrant as a point
- Qdrant builds and maintains an HNSW index over all stored vectors
- This happens once — at ingest time; nothing runs again until a query arrives

**Step 4: Query embedding (at query time):**

- The customer's email (or rewritten query — covered in Chapter 9) goes through the exact same model
- Must use the exact same model as ingest time — different model → different vector space → similarity scores mean nothing
- Produces a 384-dimensional normalised query vector

**Step 5: Approximate nearest-neighbor search:**

- The query vector is sent to Qdrant
- Qdrant's HNSW index finds the stored vectors geometrically closest to the query vector
- "Closest" = smallest angle = largest dot product (for normalised vectors)
- Returns top-k points with their payloads (text + metadata) and cosine similarity scores
- "Approximate" means HNSW may occasionally miss the single true nearest neighbor — acceptable for RAG where the second-closest chunk is almost always equally useful

**Score interpretation:**

- Score = 1.0 → vectors are identical (text is essentially the same)
- Score ~0.7–0.9 → strong semantic overlap
- Score ~0.5–0.7 → moderate semantic overlap
- Score ~0.4–0.5 → weak overlap — this is where your project's specific weakness lives
- Score < 0.3 → little to no semantic relationship

---

### 3. The Measured Weakness — Your Own Data

**The numbers from Chapter 3 (`2_Embedding_Semantic_Search.ipynb`):**

```text
FD complaint A:   "The machurity proceeds of my FD BJ2024FD4422 were supposed to come
                   to my bank account but nothing recieved yet. Where is my money?"

FD complaint B:   "Hello ji, My amnt is pending from your side since June.
                   Plz credit it fast to my bank."

Non-FD email:     "Sir ji, App me login nahi ho raha. OTP aata hi nahi.
                   Teen din se try kar raha hoon. Kya problem hai?"

similarity(FD-A, FD-B)   = 0.4528   ← same complaint, different words
similarity(FD-A, Non-FD) = 0.4135   ← completely different topic
```

**What these numbers mean:**

- The gap between "same FD complaint" and "completely different topic" is: `0.4528 - 0.4135 = 0.0393`
- That is a gap of less than 4 percentage points
- A retrieval system returning top-5 chunks needs to consistently separate relevant from irrelevant by at least this margin
- For short (31-word average), Hinglish, multi-topic emails — this gap is dangerously thin

**Comparison to Word2Vec from Chapter 0:**

```text
Word2Vec (Chapter 0, 03_Word_Embedding.ipynb):
  FD-email vs another-FD-email:  0.7598  ← much larger gap
  FD-email vs Loan-email:        0.6698
  Gap: 0.09 — more than 2× larger than the sentence transformer gap

Sentence transformer (Chapter 3):
  FD-A vs FD-B:   0.4528
  FD-A vs Non-FD: 0.4135
  Gap: 0.0393 — dangerously small
```

- This seems counterintuitive — isn't `paraphrase-multilingual-MiniLM-L12-v2` a more powerful model than Word2Vec?
- Yes, but the score range matters: sentence transformers push all scores toward the middle for short, ambiguous texts — "soft" discrimination
- The issue is not model quality; it is input quality: 31-word Hinglish emails contain too little signal for any single embedding to discriminate reliably
- This is the core reason dense retrieval alone is insufficient for your corpus and why hybrid search (Topic 4) is necessary

---

### 4. How It Is Implemented in This Project

**Chapter 3 implementation (in-memory, brute-force):**

```python
# From Chapter_3/2_Embedding_Semantic_Search.ipynb
EMBED_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(EMBED_MODEL_NAME)

def embed_texts(texts: list) -> np.ndarray:
    # batch call — one call for all texts, not one per text
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    # general formula — works even without normalisation
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Brute-force search: embed query, compare against every stored vector
def search_knowledge_base(query: str, knowledge_base: list, top_k: int = 3):
    query_vec = embed_texts([query])[0]
    scores = []
    for doc in knowledge_base:
        score = cosine_similarity(query_vec, doc["embedding"])
        scores.append((score, doc))
    return sorted(scores, key=lambda x: x[0], reverse=True)[:top_k]
```

**Chapter 6 upgrade (Qdrant — indexed, persistent):**

```python
# From Chapter_6/qdrant_*.py
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

client = QdrantClient(":memory:")  # or Docker for persistence

# At ingest time: embed and store
embeddings = model.encode([doc["page_content"] for doc in docs], normalize_embeddings=True)
points = [
    PointStruct(
        id=make_point_id(docs[i]["metadata"]["source"], i),
        vector=embeddings[i].tolist(),
        payload={"text": docs[i]["page_content"], **docs[i]["metadata"]}
    )
    for i in range(len(docs))
]
client.upsert(collection_name="fd_knowledge_base", points=points)

# At query time: embed and search
query_vector = model.encode(query, normalize_embeddings=True).tolist()
results = client.query_points(
    collection_name="fd_knowledge_base",
    query=query_vector,
    limit=5,
    with_payload=True,
).points
```

**The specific weakness this creates for FD reference numbers:**

- A query for "BJ2019FD7717" (Shobha Chopra's reference number from the test cases in Chapter 3)
- The embedding model `paraphrase-multilingual-MiniLM-L12-v2` was trained on natural language text — it has no special understanding of the `BJ####FD#####` token format
- "BJ2019FD7717" and "BJ2021FD4432" will produce vectors that are more similar to each other than either is to "premature withdrawal" — they look syntactically similar — even though they refer to completely different accounts
- This means semantic search for a specific FD reference number may return the wrong customer's record because the reference numbers embed to similar vectors
- This is exactly why `validate_fd_reference()` in Chapter 3 uses exact SQL-style lookup via `get_fd_record()`, not semantic search — exact match for exact lookups, semantic search for conceptual queries

---

### 5. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

**The thin discrimination gap — your most important production risk:**

- Measured in your data: 0.0393 gap between same-topic and different-topic emails
- At production volume of 8,000–12,000 emails/day, even a 5% misclassification rate from bad retrieval creates 400–600 wrong answers daily
- Monitor: track cosine similarity score distributions for retrieved chunks — a downward drift in the average top-1 score indicates retrieval quality degrading
- Alert: flag any retrieval result where top-1 score < 0.45 for human review — this is below even your FD-to-FD similarity threshold

**Novel product names — the "Smart Saver FD" problem:**

- This project's entire RAG motivation comes from this: a new FD product launched this week, the model has never seen its name in training
- "Smart Saver FD" produces an embedding based on its component words: "smart" + "saver" + "FD" individually
- This embedding will cluster near other savings/investment terms, but not specifically near your internal policy documents about Smart Saver FD rates and terms
- BM25 would find "Smart Saver FD" by exact term match in any document containing those exact words
- Dense retrieval's coverage of novel names depends entirely on whether those words appeared in training data with similar semantic contexts — often weak for proprietary names
- This is Topic 5 of Chapter 9 (Retrieval — Putting It to Work): testing specifically against this failure pattern

**OOV tokens and subword tokenisation:**

- `paraphrase-multilingual-MiniLM-L12-v2` uses WordPiece tokenisation — unknown words are split into subword pieces
- "BJ2019FD7717" → likely tokenised as ["BJ", "##201", "##9", "##FD", "##77", "##17"] or similar subwords
- "BJ2021FD4432" → ["BJ", "##202", "##1", "##FD", "##44", "##32"]
- The overlap in subwords ("BJ", "FD") makes these two reference numbers embed close to each other even though they refer to different accounts — a concrete embedding failure for exact identifiers

**Embedding model version drift:**

- If `paraphrase-multilingual-MiniLM-L12-v2` releases an update that slightly changes its weights, every existing vector in Qdrant becomes incomparable to new query vectors from the updated model
- The model reports the same name, but the vector space has shifted
- Your Chapter 6 incremental ingestion manifest does not track embedding model version — a known gap named explicitly there
- Fix: pin the model version (e.g. `sentence-transformers==2.x.x`) in requirements.txt, and when upgrading, trigger a full re-embed

**Latency (concrete numbers for your setup):**

- Model load time: ~3–5 seconds on first call (model downloads cached to disk), then ~0.1–0.5 seconds per subsequent load
- Embedding one 31-word email on CPU: ~20–50ms
- Embedding one 31-word email on GPU (RTX 4060): ~2–5ms
- Qdrant HNSW search over 17 pages in `:memory:` mode: under 1ms
- Total query path: ~20–50ms CPU or ~2–5ms GPU — dominated by the embedding call, not the vector search
- At 8,000–12,000 emails/day: ~93 emails/minute → well within CPU capacity; no GPU needed at this volume

**Cost:**

- `paraphrase-multilingual-MiniLM-L12-v2` runs locally — zero per-query API cost
- Memory footprint: ~90MB for the model weights + 384 × 4 bytes per stored vector = negligible for 17 pages
- At 100,000 pages: 384 × 4 × 100,000 = ~150MB for vectors alone — still very manageable
- Compare to OpenAI `text-embedding-ada-002`: ~$0.0001 per 1K tokens → at 8,000 emails/day × 31 words/email ≈ 250K tokens/day ≈ $0.025/day → $9/year; local model saves this but requires compute

**Monitoring — what to track:**

- Distribution of top-1 cosine similarity scores across all queries — downward drift = retrieval degrading
- Fraction of queries where top-1 and top-2 scores are within 0.02 of each other — high fraction = borderline cases, retrieval is uncertain
- Embedding latency percentiles (p50, p95, p99) — spikes indicate CPU/GPU contention or model reload
- Fraction of queries where BM25 top-5 and dense top-5 overlap — measures retrieval diversity; should inform hybrid weighting

**Security:**

- Embedding vectors can leak information about the original text in theory (vector inversion attacks exist in research literature)
- For your corpus with PII in customer records (Mobile_Number, Account_Number from `fd_master_database.csv`), embedding those records and storing in Qdrant creates a secondary copy of PII in vector form
- Covered in detail in Chapter 6 Topic 9 (PII and Access-Scoping) — apply caller-scoped payload filtering so cross-customer semantic matches cannot return another customer's records
- The embedding model itself runs locally — no data leaves your machine during embedding, unlike API-based embedding services where text is transmitted to external servers

**Deployment:**

- For 8,000–12,000 emails/day on CPU: a single process with the model loaded once at startup handles the volume; no GPU needed
- For lower latency requirements (real-time response < 100ms): switch to GPU inference on your RTX 4060 (2–5ms per embed vs 20–50ms on CPU)
- For higher throughput: batch queries together (embed 32 at a time vs 1 at a time) — the model is significantly more efficient in batch mode due to matrix operation parallelism

---

### 6. Design Decisions and Trade-offs

**Why this model specifically (`paraphrase-multilingual-MiniLM-L12-v2`):**

- Multilingual: covers both English (policy documents) and Hinglish (customer emails) in one model — critical for your corpus
- 384 dimensions: small enough for fast search and low memory, large enough to capture meaningful semantic distinctions
- "Paraphrase" training objective: explicitly trained to produce similar embeddings for paraphrases — exactly the use case of matching semantically equivalent but differently-worded texts
- Free, local, no API cost: aligned with the project's design philosophy (Anthropic handles generation, local open-source handles embedding)
- Limitation: it is not fine-tuned on FD domain text — "byaj" and "interest" may not be as close as they should be in your domain

**Mean pooling vs CLS token:**

- Chapter 0's BERT/MuRIL notes covered the `[CLS]` token: a special token whose final hidden state summarises the whole input, used for classification
- This embedding model uses mean pooling instead: average all token embeddings at the final layer
- Mean pooling generally produces better sentence embeddings for retrieval (preserves contributions from all words, not just the classification token)
- The `[CLS]` approach in Chapter 0's MuRIL works for classification because the fine-tuning process aligns `[CLS]` to the task; for general-purpose retrieval without task-specific fine-tuning, mean pooling is better

**Brute-force in-memory (Chapter 3) vs HNSW in Qdrant (Chapter 6):**

- Brute-force guarantees exact nearest neighbors — no misses; Qdrant HNSW is approximate — occasional misses
- Brute-force is O(n) per query — scales linearly with corpus size; Qdrant HNSW is approximately O(log n) — scales much better
- Brute-force has no persistence — all vectors lost on restart; Qdrant persists to disk (with Docker volume mount)
- Brute-force has no metadata filtering during search; Qdrant filters during HNSW traversal
- At 17 pages: both are fast, brute-force is simpler; at 17,000 pages: Qdrant HNSW is necessary

**Why not fine-tune the embedding model on FD domain text?**

- Fine-tuning would improve discrimination for FD-specific terms ("byaj", "machurity", "naya fd" mapped closer to their English equivalents)
- But requires labeled pairs: (query, positive_document, negative_document) — expensive to create
- The benefit needs to be measured against the cost of maintaining a custom model (re-training when the base model updates, storage, inference infrastructure)
- The right trigger: measure BM25 + dense hybrid recall first; if still insufficient after query expansion and reranking, then consider domain fine-tuning
- Chapter 18 (Fine-Tuning) covers this in full for the generation side; the same decision framework applies to embedding model fine-tuning

---

### 7. Alternatives and When to Use Each

**Local sentence-transformer models (this project's choice):**

- `paraphrase-multilingual-MiniLM-L12-v2`: multilingual, 384-dim, free, fast — the right choice for your Hinglish + English corpus at this stage
- `all-MiniLM-L6-v2`: English-only, 384-dim, faster — would fail on Hinglish emails
- `multilingual-e5-large`: 1024-dim, higher quality, much slower — worth evaluating if current model's 0.0393 gap proves too narrow after hybrid search
- `paraphrase-multilingual-mpnet-base-v2`: 768-dim, better quality than MiniLM, ~2× slower

**API-based embedding models:**

- OpenAI `text-embedding-3-small`: 1536-dim, strong multilingual coverage, $0.00002/1K tokens — worth evaluating for quality improvement at low cost
- OpenAI `text-embedding-3-large`: 3072-dim, highest quality, more expensive — only if smaller model proves insufficient
- Anthropic does not offer an embedding model — Claude handles generation only; local or third-party models handle embedding (this is explicitly noted in Chapter 3)
- Cohere `embed-multilingual-v3`: strong multilingual embedding, good for Hinglish, API-based

**Specialised retrieval models:**

- `DPR` (Dense Passage Retrieval, Meta): two separate encoders — one for queries, one for documents (asymmetric); better when query and document styles differ significantly; covered in Topic 3
- `ColBERT`: late interaction model; produces one vector per token (not one per document); much more expressive but also much more expensive to store and search; covered in Topic 3
- `E5` family: explicitly trained with "query:" and "passage:" prefixes to distinguish query-side vs document-side encoding — better for RAG than symmetric models like `paraphrase-*`

---

### 8. Common Mistakes and Production Failures

- **Using different models at ingest and query time** — the most catastrophic failure mode: vectors in Qdrant from Model A are incomparable to query vectors from Model B; retrieval returns random results with no error; detected only by monitoring top-1 score distribution dropping
- **Not normalising vectors** — `normalize_embeddings=False` means dot product ≠ cosine similarity; longer texts produce longer vectors and score higher regardless of relevance; always normalise
- **Embedding one text at a time in a loop** — `for text in texts: model.encode(text)` is 10–50× slower than `model.encode(texts)` for a batch; the model is designed for batch processing via matrix operations
- **Semantic search for exact-match queries** — querying for "BJ2019FD7717" via dense retrieval instead of via `validate_fd_reference()` and `get_fd_record()` will return semantically similar but wrong records; exact identifiers need exact-match lookup, not semantic similarity
- **Not monitoring the similarity score distribution** — the 0.0393 gap in your corpus means borderline retrieval failures are invisible until answer quality visibly degrades; track scores continuously, not just accuracy metrics
- **Treating dense retrieval as infallible for novel terms** — a new FD product ("Smart Saver FD") launched after embedding model training will have a weak, context-insensitive embedding; BM25 will find it by exact term match; this is why hybrid search is essential
- **Ignoring the multilingual requirement** — using an English-only embedding model (`all-MiniLM-L6-v2`) on your 64.4% Hinglish corpus will collapse all Hinglish text to similar vectors near whatever the nearest English words are; never use English-only models on a multilingual corpus

---

### 9. Lead-Level Interview Questions

**Basic:**

**Q: What is the difference between sparse retrieval (BM25) and dense retrieval?**

A: BM25 matches words literally — it finds documents containing the same words as the query, weighted by rarity and frequency. Dense retrieval matches meaning — it maps both query and documents to vector spaces where semantic similarity corresponds to geometric proximity, and finds documents whose meaning is close to the query regardless of word choice. BM25 scores zero when there is no vocabulary overlap; dense retrieval can score positively even when there is zero word overlap, if the meanings are related.

**Q: Why does your project use `normalize_embeddings=True` and what does this enable?**

A: Normalising vectors to unit length makes cosine similarity equal to the dot product. Cosine similarity formula is `dot(a,b) / (||a|| × ||b||)`. When `||a|| = ||b|| = 1`, this reduces to just `dot(a,b)`. This matters for two reasons: computation is cheaper (no division needed), and Qdrant's internal similarity computations use this same shortcut, so normalised vectors are the correct input for `distance=Distance.COSINE`. Your Chapter 3 notebook confirmed this empirically with a sanity-check print.

**Intermediate:**

**Q: Your Chapter 3 notebook measured cosine similarity of 0.4528 between two FD emails and 0.4135 between an FD email and a completely unrelated email. What does this tell you about your retrieval system?**

A: It tells me the discrimination gap is dangerously thin — only 0.0393 separates "same complaint" from "completely different topic" for short Hinglish emails. Any retrieval system that needs to consistently distinguish FD-relevant from FD-irrelevant content has only a 4% margin of semantic separation to work with. This means: dense retrieval alone is insufficient for this corpus, threshold-based filtering (return results above 0.5 similarity) will miss many relevant results and include many irrelevant ones, and hybrid search combining BM25's exact-match strength with dense retrieval's semantic strength is necessary. This is the quantitative evidence that motivated building hybrid search rather than relying on dense retrieval alone.

**Q: Why would semantic search for an FD reference number like "BJ2019FD7717" fail, and how does your project handle this?**

A: The embedding model tokenises "BJ2019FD7717" into subword pieces — "BJ", "2019", "FD", "7717" — and embeds it based on those pieces' semantic context in training data. "BJ2021FD4432" (a different customer's account) shares the "BJ" and "FD" subwords, so both reference numbers embed to geometrically similar vectors. Semantic search for one would return results associated with the other — cross-customer data leakage. Chapter 3 handles this by using `validate_fd_reference()` → `get_fd_record()` for exact-match lookups, which is a SQL-style lookup by primary key, not semantic search. The rule of thumb: exact identifiers and codes always use exact-match lookup; conceptual questions use semantic search.

**Advanced:**

**Q: Your embedding model was trained on general multilingual text. What are the specific ways this creates retrieval failures for FD domain content, and what would you do about it?**

A: Three failure modes. First, domain vocabulary: "byaj" and "interest" mean the same thing in your corpus but the embedding model may not have seen "byaj" frequently enough in financial contexts to place it close to "interest" in vector space — fix with domain fine-tuning using contrastive pairs (byaj, interest) or with query expansion preprocessing. Second, Hinglish code-mixing: the model handles both Hindi and English but may not handle their interleaving as well as either language alone — a sentence mixing both may embed ambiguously; hard to fix without a Hinglish-specific fine-tuned model. Third, novel product names: any FD product launched after the model's training cutoff will not have a trained semantic relationship to relevant policy text — this is unfixable by the embedding model; BM25 covers it by exact term match. The practical fix order is: measure current recall with hybrid BM25 + dense, then add query expansion for known Hinglish synonyms, then consider domain fine-tuning only if measured recall is still insufficient after those cheaper fixes.

**Q: A teammate argues that since you're using Qdrant's HNSW approximate nearest-neighbor search, you might be missing relevant chunks that exact brute-force search would find. How do you respond?**

A: The tradeoff is real but the impact is acceptable for RAG. HNSW has high recall (typically 95–99%) at the k values used in RAG (top-5, top-10). The chunks it occasionally misses are almost always the second or third-best matches, not the best match, because HNSW traversal is designed to find the closest neighbors reliably. For a RAG system, having 5 highly relevant chunks where 1 might occasionally be replaced by the 6th-best chunk rather than the 5th-best is negligible compared to the latency and scalability gains HNSW provides over brute-force. The real retrieval failures in your corpus come from the 0.0393 discrimination gap and vocabulary mismatch — not from HNSW's approximate nature. If exact guarantees were needed (fraud detection, legal document matching), that would require brute-force exact search.

**Scenario-based:**

**Q: After deploying your RAG system, you notice that answers about a newly launched "Bajaj Finserv Flexi-Rate FD" product are always wrong or missing. Walk through your diagnosis.**

A: Start with the retrieval layer, not the generation layer. Check whether "Flexi-Rate FD" documents were ingested into the knowledge base — run `load_all_json()` and verify sources. If ingested, check BM25 scores for a "Flexi-Rate FD" query against those chunks — BM25 should find them by exact term match. If BM25 finds them but dense retrieval does not, the problem is that the embedding model has no training signal for "Flexi-Rate FD" and embeds it to a general savings/investment region of vector space that may not be close to the specific policy chunks. Fix: rely on BM25 for novel product name retrieval (this is exactly what BM25 is better at), and ensure hybrid search weights BM25 strongly enough that exact-match queries are not drowned out by dense retrieval's semantic blurriness. If BM25 also fails, the documents were not ingested — check the incremental ingestion manifest to confirm the new policy files were processed.

---

### 10. Hidden Concepts and Prerequisites

**Mean pooling vs CLS token — why it matters:**

- BERT-family models produce one vector per input token
- To get one vector per sentence, you need to aggregate: take `[CLS]` token's vector (BERT's original approach) or average all token vectors (mean pooling)
- Mean pooling for retrieval was empirically shown to outperform `[CLS]` in the SBERT paper (Reimers & Gurevych, 2019) — it preserves signal from all words equally
- `paraphrase-multilingual-MiniLM-L12-v2` uses mean pooling (confirmed by the model architecture print in Chapter 3: `Pooling({'pooling_mode': 'mean'})`)
- Chapter 0's MuRIL uses `[CLS]` for classification — different task, different pooling strategy, both correct for their respective uses

**The curse of dimensionality and why it does not kill dense retrieval:**

- In very high-dimensional spaces, distances between random points converge — most points become "equally far" from any query, making distance a useless discriminator
- 384 dimensions is high-dimensional, yet dense retrieval works well — why?
- Because the embedding model is not placing vectors randomly: it learns a structured manifold where semantically related texts cluster together
- The learned structure means distances are not random — the high-dimensionality curse applies to random vectors, not to learned embeddings
- This is also why cosine similarity (angle-based) works better than Euclidean distance (magnitude-based) for embeddings: magnitude in embedding space is not semantically meaningful, but direction is

**Bi-encoders vs cross-encoders:**

- Your current setup is a bi-encoder: query and document are embedded independently, similarity is computed after
- A cross-encoder takes query and document together as input and outputs a single relevance score — much more accurate (the model sees the interaction between query and document), much slower (must run for every (query, document) pair at search time)
- This is the foundation of reranking (Topic 7): use a bi-encoder (fast) to retrieve top-k candidates, then use a cross-encoder (accurate) to rerank only those k candidates
- The bi-encoder/cross-encoder distinction also maps to the DPR and ColBERT architectures covered in Topic 3

**Semantic similarity vs lexical similarity — the full picture:**

- BM25 measures lexical similarity: how much do the words overlap, weighted by rarity?
- Dense retrieval measures semantic similarity: how close are the meanings in learned vector space?
- They are complementary, not competing: a question about "premature withdrawal" should match both a document saying "premature withdrawal" (lexical) and a document saying "early closure" (semantic)
- No single retrieval method captures both well — hybrid search (Topic 4) combines them

---

### 11. Revision Summary

> Dense retrieval maps both queries and documents to fixed-size vectors in a learned semantic space, enabling retrieval based on meaning rather than word overlap. Your project already built this in Chapter 3 using `paraphrase-multilingual-MiniLM-L12-v2` (384-dim, multilingual, local). The specific weakness measured in your own data: cosine similarity gap between same-topic and different-topic short Hinglish emails is only 0.0393 (0.4528 vs 0.4135) — dangerously thin for reliable standalone retrieval. Dense retrieval also fails specifically on: exact identifiers (BJ2019FD7717 — reference numbers embed to similar vectors regardless of which account they refer to, handled by `validate_fd_reference()` using exact-match lookup), and novel product names (embeddings for unseen product names like "Smart Saver FD" are weak and imprecise). The architecture answer to all three weaknesses is hybrid search combining BM25 (exact match, novel terms) and dense retrieval (vocabulary mismatch, semantic paraphrase) — covered in Topic 4. Never use different embedding models at ingest and query time; always normalise vectors; always batch-embed rather than one-at-a-time; and always use exact-match lookup for exact identifiers, not semantic search.

---
